In [1]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt

In [2]:
df = pd.read_csv("../data/processed/poi_raw_malang.csv")
print(f"Loaded Data: {len(df)} row")
print(f"Categories: {df['category'].value_counts().to_dict()}")

Loaded Data: 1554 row
Categories: {'place_of_worship': 633, 'school': 447, 'residential': 327, 'park': 58, 'marketplace': 31, 'college': 30, 'university': 28}


In [3]:
CATEGORY_MAP = {
    "n_campus":            ["university", "college"],
    "n_school":            ["school"],
    "n_market":            ["marketplace"],
    "n_place_of_worship":  ["place_of_worship"],
    "n_park":              ["park", "playground"],
    "n_residential":       ["residential"],
}

In [4]:
import h3
from shapely.geometry import Polygon

# Bounding box Malang City
LAT_MIN = -8.0400   #(paling selatan — Kedungkandang bawah)
LAT_MAX = -7.9200   #(paling utara  — Lowokwaru atas)
LON_MIN = 112.5650  #(paling barat  — Lowokwaru kiri)
LON_MAX = 112.6940  #(paling timur  — Blimbing kanan)

polygon = h3.LatLngPoly([
    (LAT_MIN, LON_MIN),
    (LAT_MIN, LON_MAX),
    (LAT_MAX, LON_MAX),
    (LAT_MAX, LON_MIN),
    (LAT_MIN, LON_MIN),  
])

cells = h3.polygon_to_cells(polygon, res=9)

grid_rows = []
for cell in cells:
    lat, lon = h3.cell_to_latlng(cell)
    grid_rows.append({
        "grid_id": cell,
        "lat" : lat,
        "lon" : lon
    })

grid_df = pd.DataFrame(grid_rows)
print(f"Generated grid with {len(grid_df)} cells (500m x 500m)")


Generated grid with 1969 cells (500m x 500m)


In [5]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # earth radius in meters
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * asin(sqrt(a))

In [6]:
RADIUS_M = 500 # 500 meter radius

for col in CATEGORY_MAP:
    grid_df[col] = 0

poi_by_group = {}
for col, categories in CATEGORY_MAP.items():
    poi_by_group[col] = df[df["category"].isin(categories)].copy()

for idx, row in grid_df.iterrows():
    if idx % 100 == 0:
        print(f"Processing grid {idx+1}/{len(grid_df)}")
    
    for col, poi_subset in poi_by_group.items():
        if poi_subset.empty:
            continue

        distances = poi_subset.apply(
            lambda p: haversine(row["lat"], row["lon"], p["lat"], p["lon"]),
            axis=1
        )
        grid_df.at[idx, col] = int((distances <= RADIUS_M).sum())

print("\fFinished calculating all grids")

Processing grid 1/1969


Processing grid 101/1969
Processing grid 201/1969
Processing grid 301/1969
Processing grid 401/1969
Processing grid 501/1969
Processing grid 601/1969
Processing grid 701/1969
Processing grid 801/1969
Processing grid 901/1969
Processing grid 1001/1969
Processing grid 1101/1969
Processing grid 1201/1969
Processing grid 1301/1969
Processing grid 1401/1969
Processing grid 1501/1969
Processing grid 1601/1969
Processing grid 1701/1969
Processing grid 1801/1969
Processing grid 1901/1969
Finished calculating all grids


In [7]:
# dominant zone classification
def classify_zone(row):
    scores = {
        "campus_area"      : row["n_campus"] * 5,
        "school_area"      : row["n_school"] * 2,
        "residential_area" : row["n_residential"] * 3,
        "commercial_area"  : row["n_market"] * 4,
        "worship_area"     : row["n_place_of_worship"] * 1,
        "public_area"      : row["n_park"] * 3,
    }

    max_score = max(scores.values())
    if max_score == 0:
        return "not classified"
    return max(scores, key=scores.get)

grid_df["dominant_zone"] = grid_df.apply(classify_zone, axis=1)

In [8]:
count_cols = list(CATEGORY_MAP.keys())
grid_filtered = grid_df[grid_df[count_cols].sum(axis=1) > 0].copy()
 
print(f"\nGrids with at least 1 POI: {len(grid_filtered)} out of {len(grid_df)} total")
print(f"\nDominant zone distribution:")
print(grid_filtered["dominant_zone"].value_counts())


Grids with at least 1 POI: 1186 out of 1969 total

Dominant zone distribution:
dominant_zone
residential_area    513
school_area         363
worship_area        181
campus_area          88
public_area          32
commercial_area       9
Name: count, dtype: int64


In [9]:
grid_df.to_csv("../data/processed/poi_grid_malang.csv", index=False)
 
grid_filtered.to_csv("../data/processed/poi_grid_malang_filtered.csv", index=False)
 
print(f"\n✅ Saved:")
print(f"   data/processed/poi_grid_malang.csv          ({len(grid_df)} rows)")
print(f"   data/processed/poi_grid_malang_filtered.csv ({len(grid_filtered)} rows)")
print(f"\nPreview first 5 rows (filtered):")
print(grid_filtered.head().to_string())


✅ Saved:
   data/processed/poi_grid_malang.csv          (1969 rows)
   data/processed/poi_grid_malang_filtered.csv (1186 rows)

Preview first 5 rows (filtered):
           grid_id       lat         lon  n_campus  n_school  n_market  n_place_of_worship  n_park  n_residential     dominant_zone
1  898d85d0c6fffff -8.017185  112.660519         0         0         0                   0       0              1  residential_area
3  898d85d0e33ffff -8.007544  112.666335         0         0         0                   1       0              1  residential_area
4  898d85d52bbffff -7.995517  112.600222         0         1         0                   8       0              9  residential_area
5  898d85d0aabffff -7.995339  112.654138         0         2         0                   8       0              0      worship_area
6  898d85d5643ffff -7.997199  112.621225         0         5         1                   6       0              2       school_area
